## Cox Survival Analysis by HF Cluster

Evaluates time from first HF diagnosis to inpatient visits, emergency-room visits, and either outcome using age-quartile-stratified Cox models.

### Inputs
- `Outputs/vade_assignments/patient_cluster_assignments_jhu_transfer.parquet`: Patient cluster assignments.
- `unique_patient_ids_with_hf_date.csv`: Patient IDs and first HF diagnosis dates.

### Outputs
- **Figures saved under `Figures/`:**
  - Hazard-ratio forest plots for each outcome.
  - Kaplan–Meier curves by cluster.
  - Follow-up time histograms.
  - Combined forest plot: `forestplot_HF_combined.png`.

In [ ]:
import os
import numpy as np
import pandas as pd
import pandas_gbq
import matplotlib.pyplot as plt
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.plotting import add_at_risk_counts

In [ ]:
cluster_df = pd.read_csv('gs://workspace-bucket-wb-frigid-jalapeno-2803/outputs/s3vdc-vade-aou-fixedc4-deaths-20261009T041812Z-5d3782cf/trial-19/assignments.csv')

In [ ]:
hf_dates = pd.read_csv("gs://workspace-bucket-wb-frigid-jalapeno-2803/unique_patient_ids_with_hf_date_deaths.csv")

In [ ]:
hf_dates.shape

In [ ]:
cluster_df.shape

In [ ]:
hf_dates = hf_dates.rename(columns={"PERSON_ID": "person_id"})
cluster_df["person_id"] = cluster_df["person_id"].astype(int)
hf_dates["person_id"] = hf_dates["person_id"].astype(int)
hf_dates["first_hf_diagnosis_date"] = pd.to_datetime(
    hf_dates["first_hf_diagnosis_date"], format="mixed", utc=True
).dt.tz_localize(None)

In [ ]:
cluster_df["cluster"] = cluster_df["cluster"] - cluster_df["cluster"].min() + 1
N_CLUSTERS = int(cluster_df["cluster"].nunique())

CLUSTER_NAMES = {
    1: "Hypertensive Pregnancy Analog",
    2: "Cardiorenal Analog",
    3: "Autoimmune Analog",
    4: "Ischemia,T2DM Analog",
}

cluster_labels = {f"cluster_{k}": v for k, v in CLUSTER_NAMES.items()}

dummies = pd.get_dummies(cluster_df["cluster"], prefix="cluster").astype(int)
clus_members = pd.concat([cluster_df[["person_id"]], dummies], axis=1)

cluster_cols = [f"cluster_{i}" for i in range(1, N_CLUSTERS + 1)]
 
cohort = (
    hf_dates.merge(clus_members, on="person_id", how="inner")
    .rename(columns={"first_hf_diagnosis_date": "index_date"})
)
print(f"Cohort: {len(cohort)} patients")
print(cohort[cluster_cols].sum())

In [ ]:
DATASET = "wb-silky-artichoke-2408.C2024Q3R9"

In [ ]:
GOOGLE_PROJECT = os.environ["GOOGLE_PROJECT"]
pandas_gbq.context.project = GOOGLE_PROJECT

In [ ]:
import os
os.makedirs("./Figures_Cox_copy", exist_ok=True)
os.makedirs("./Outputs_Cox_copy", exist_ok=True)

In [ ]:
def _chunks(lst, n=10000):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]

def read_gbq_by_ids(table_cols, table, ids, extra_where=""):
    frames = []
    for batch in _chunks(ids):
        id_str = ", ".join(str(p) for p in batch)
        sql = f"""
        SELECT {table_cols}
        FROM `{DATASET}.{table}`
        WHERE person_id IN ({id_str}) {extra_where}
        """
        frames.append(pandas_gbq.read_gbq(sql, progress_bar_type=None))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

ids = cohort["person_id"].tolist()

visits = read_gbq_by_ids("person_id, visit_concept_id, visit_start_date", "visit_occurrence", ids, extra_where="AND visit_concept_id IN (9201, 9203)")
visits["person_id"] = visits["person_id"].astype(int)
visits["visit_start_date"] = pd.to_datetime(visits["visit_start_date"])

obs = read_gbq_by_ids("person_id, observation_period_end_date", "observation_period", ids)
obs["person_id"] = obs["person_id"].astype(int)
obs["observation_period_end_date"] = pd.to_datetime(obs["observation_period_end_date"])
obs = obs.groupby("person_id", as_index=False)["observation_period_end_date"].max()

person = read_gbq_by_ids("person_id, birth_datetime","person",ids)
person["person_id"] = (person["person_id"].astype("int64"))
person["birth_date"] = (pd.to_datetime(person["birth_datetime"],utc=True,errors="coerce").dt.tz_localize(None).dt.normalize())

base = (cohort.merge(obs, on="person_id", how="left").merge(person[["person_id", "birth_date"]],on="person_id", how="left"))
base["age"] = ((pd.to_datetime(base["index_date"]) -pd.to_datetime(base["birth_date"])).dt.days / 365.25)

# age as a stratum via quartiles, computed once on the full cohort
base["age_quartile"] = pd.qcut(base["age"], q=4).astype(str)
base.loc[base["age"].isna(), "age_quartile"] = np.nan

In [ ]:
from matplotlib.ticker import ScalarFormatter

def run_outcome(visit_concept_ids, label, file_tag):
    print(f"\n{label}")

    ev = visits[visits["visit_concept_id"].isin(visit_concept_ids)]
    df = base.merge(ev[["person_id", "visit_start_date"]], on="person_id", how="left")
    df = df[df["visit_start_date"] > df["index_date"]]
    first_event = (df.groupby("person_id")["visit_start_date"].min().reset_index().rename(columns={"visit_start_date": "event_date"}))
    df = base.merge(first_event, on="person_id", how="left")

    df["end_date"] = df["event_date"].fillna(df["observation_period_end_date"])
    df["duration"] = (df["end_date"] - df["index_date"]).dt.days
    df["has_event"] = df["event_date"].notna().astype(int)
    df = df.dropna(subset=["age_quartile", "duration"])
    df = df[df["duration"] > 0].reset_index(drop=True)
    df["followup_years"] = df["duration"] / 365.25

    clus_size = df[cluster_cols].sum()
    ref_cluster = clus_size.idxmax()
    covariates = [c for c in cluster_cols if c != ref_cluster]

    print("Reference cluster:", ref_cluster)
    print("Covariates:", covariates)

    df_mod = df[["duration", "has_event", "age_quartile"] + covariates].reset_index(drop=True)

    cph = CoxPHFitter()
    formula = " + ".join(covariates)
    cph.fit(df_mod, duration_col="duration", event_col="has_event",
            formula=formula, strata=["age_quartile"])
    print(f"Reference cluster: {ref_cluster} | n={len(df_mod)} | events={int(df_mod['has_event'].sum())}")
    cph.print_summary()

    try:
        cph.check_assumptions(df_mod, p_value_threshold=0.05, show_plots=True)
    except Exception as e:
        print("check_assumptions skipped:", e)

    summ = cph.summary.reset_index().rename(columns={"covariate": "term"})
    rows = [{
        "cluster": r["term"], "hr": r["exp(coef)"],
        "lower": r["exp(coef) lower 95%"], "upper": r["exp(coef) upper 95%"],
        "pval": round(r["p"], 3), "case": label,
    } for _, r in summ[summ["term"].isin(cluster_cols)].iterrows()]
    rows.append({"cluster": ref_cluster, "hr": 1.0, "lower": 1.0, "upper": 1.0,
                 "pval": "ref", "case": label})
    forest_df = pd.DataFrame(rows)

    order = cluster_cols
    fdf = forest_df.set_index("cluster").reindex(order).reset_index()
    figf, axf = plt.subplots(figsize=(8, 0.7 * len(order) + 1.5))
    for i, r in fdf.iterrows():
        if pd.isna(r["hr"]):
            continue
        axf.plot([r["lower"], r["upper"]], [i, i], color="darkblue", lw=2)
        axf.plot(r["hr"], i, "s", color="royalblue", ms=9)
        axf.text(2.55, i, f"HR {r['hr']:.2f}   p={r['pval']}", va="center", fontsize=9)
    axf.axvline(1.0, color="grey", ls="--", lw=1)
    axf.set_yticks(range(len(order)))
    axf.set_yticklabels([
    cluster_labels[c] + " (ref)" if c == ref_cluster else cluster_labels[c]
    for c in order
])
    axf.set_xscale("log")
    axf.set_xlim(0.5, 2.5)
    axf.set_xticks([0.5, 0.6, 0.7, 1.0, 1.5, 2.0, 2.5])
    axf.xaxis.set_major_formatter(ScalarFormatter())
    axf.set_xlabel("Hazard Ratio (log scale)")
    axf.set_title(label)
    plt.tight_layout()
    plt.savefig(f"./Figures_Cox_copy/forest_{file_tag}.png", dpi=300, bbox_inches="tight")
    plt.show()

    # Kaplan-Meier
    fig, ax = plt.subplots(figsize=(10, 8))
    kmf_list = []
    for c in cluster_cols:
        group = df[df[c] == 1]
        if group.empty:
            continue
        kmf = KaplanMeierFitter()
        kmf.fit(group["duration"] / 365, group["has_event"],
                label=f"{cluster_labels[c]} (ref)" if c == ref_cluster else cluster_labels[c])
        kmf_list.append(kmf)
        kmf.plot(ax=ax, linewidth=2)
    ax.set_xlim(0, 7)
    ax.set_ylim(-0.1, 1.1)
    ax.set_xlabel(f"Years from first {label} visit")
    ax.set_ylabel("Risk-free probability")
    ax.set_title(label)
    plt.tight_layout()
    plt.savefig(f"./Figures_Cox_copy/cox_kap_{file_tag}.png", dpi=300)
    plt.show()

    # Follow-up time histogram
    print(f"Mean follow-up (years): {df['followup_years'].mean():.2f}")

    fig_hist, ax_hist = plt.subplots(figsize=(8, 5))
    df["followup_years"].hist(bins=30, edgecolor="black", ax=ax_hist)
    ax_hist.set_xlabel("Follow-up time (years)")
    ax_hist.set_ylabel("Number of patients")
    ax_hist.set_title(f"Distribution of Follow-up Time ({label})")
    plt.tight_layout()
    plt.savefig(f"./Figures_Cox_copy//followup_{file_tag}.png", dpi=300)
    plt.show()

    return cph, forest_df

In [ ]:
cph_inpatient,forest_inpatient = run_outcome([9201], "Inpatient", "9201")

In [ ]:
cph_er,forest_er = run_outcome([9203], "ER", "9203")

In [ ]:
cph_both,forest_both = run_outcome([9201, 9203], "Inpatient + ER", "both")

In [ ]:
from matplotlib.patches import Patch

plot_all_forest = pd.concat([forest_inpatient, forest_er, forest_both], ignore_index=True)
d = plot_all_forest.copy()

XMAX = 4
LEFT, LABEL_X, PVAL_X = -1.10, -1.08, -0.02
cases = ["Inpatient", "ER", "Inpatient + ER"]
colors = {"Inpatient": "#2f7df6", "ER": "#ff0000", "Inpatient + ER": "#2e8b2e"}
offsets = {"Inpatient": -0.22, "ER": 0.0, "Inpatient + ER": 0.22}

display_to_key = {"Autoimmune Analog": "cluster_3", "Cardiorenal Analog": "cluster_2", "Hypertensive Pregnancy Analog (ref)": "cluster_1", "Ischemia,T2DM Analog": "cluster_4"}
display_order = list(display_to_key)
y_pos = {label: i for i, label in enumerate(display_order)}

d["lower"] = pd.to_numeric(d["lower"], errors="coerce")
d["upper"] = pd.to_numeric(d["upper"], errors="coerce")
d["hr"] = pd.to_numeric(d["hr"], errors="coerce")
d["pval"] = pd.to_numeric(d["pval"], errors="coerce")
d["lower_plot"] = d["lower"].clip(lower=1e-6, upper=XMAX)
d["upper_plot"] = d["upper"].clip(lower=1e-6, upper=XMAX)
d["hr_plot"] = d["hr"].clip(lower=1e-6, upper=XMAX)
d["se"] = (np.log(d["upper_plot"]) - np.log(d["lower_plot"])) / (2 * 1.96)
d["w"] = 1 / d["se"] ** 2

ref_key = display_to_key["Hypertensive Pregnancy Analog (ref)"]
wmax = d.loc[d["cluster"] != ref_key, "w"].replace([np.inf, -np.inf], np.nan).max()

def msize(w):
    return float(np.clip(14 * np.sqrt(w / wmax), 3, 14)) if pd.notna(w) and pd.notna(wmax) and wmax > 0 else 7

def fmt_p(p):
    return "<0.001" if pd.notna(p) and p < 0.001 else f"{p:.3f}" if pd.notna(p) else "NA"

fig, ax = plt.subplots(figsize=(14, 6))
fig.subplots_adjust(left=0.42, right=0.80, bottom=0.14, top=0.90)

for label in ["Cardiorenal Analog", "Ischemia,T2DM Analog"]:
    ax.axhspan(y_pos[label] - 0.5, y_pos[label] + 0.5, xmin=LEFT, xmax=1, color="#f1f5f8", zorder=0, clip_on=False)

cap = 0.07

for label in display_order:
    ax.text(LABEL_X, y_pos[label], label, transform=ax.get_yaxis_transform(), ha="left", va="center", fontsize=13)

    for case in cases:
        y = y_pos[label] + offsets[case]

        if "(ref)" in label:
            ax.plot(1, y, "s", color=colors[case], markersize=12, zorder=3)
            ax.text(PVAL_X, y, "ref", transform=ax.get_yaxis_transform(), ha="right", va="center", fontsize=11)
            continue

        row = d[(d["case"] == case) & (d["cluster"] == display_to_key[label])].iloc[0]
        lo, hi, hr = row["lower_plot"], row["upper_plot"], row["hr_plot"]

        if pd.isna(lo) or pd.isna(hi) or pd.isna(hr):
            ax.text(PVAL_X, y, "NA", transform=ax.get_yaxis_transform(), ha="right", va="center", fontsize=11)
            continue

        if row["upper"] > XMAX or not np.isfinite(row["upper"]):
            ax.annotate("", xy=(XMAX, y), xytext=(lo, y), arrowprops=dict(arrowstyle="->", color="black", lw=1.6, shrinkA=0, shrinkB=0))
            ax.plot([lo, lo], [y - cap, y + cap], color="black", lw=1.6)
        else:
            ax.plot([lo, hi], [y, y], color="black", lw=1.6, zorder=2)
            ax.plot([lo, lo], [y - cap, y + cap], color="black", lw=1.6)
            ax.plot([hi, hi], [y - cap, y + cap], color="black", lw=1.6)

        ax.plot(hr, y, "s", color=colors[case], markersize=msize(row["w"]), zorder=3)
        ax.text(PVAL_X, y, fmt_p(row["pval"]), transform=ax.get_yaxis_transform(), ha="right", va="center", fontsize=11)

ax.text(LABEL_X, -0.75, "Cluster", transform=ax.get_yaxis_transform(), fontsize=15, fontweight="bold", va="center")
ax.text(PVAL_X, -0.75, "p-value", transform=ax.get_yaxis_transform(), fontsize=15, fontweight="bold", va="center", ha="right")
ax.set_xlim(0, XMAX)
ax.set_ylim(len(display_order) - 0.5, -0.5)
ax.set_xticks(np.arange(0, XMAX + 0.5, 0.5))
ax.set_xticklabels([f"{x:g}" for x in np.arange(0, XMAX + 0.5, 0.5)])
ax.tick_params(axis="x", labelsize=13)
ax.set_xlabel("Hazard Ratios", fontsize=16)
ax.set_yticks([])

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

handles = [Patch(facecolor=colors[case], edgecolor="none", label=case) for case in cases]
ax.legend(handles=handles, loc="center left", bbox_to_anchor=(1.03, 0.5), frameon=False, prop={"size": 13, "weight": "bold"})

plt.savefig("./Figures_Cox_copy//forestplot_HF_combined.png", dpi=300, bbox_inches="tight")
plt.show()